# LMI — Google Colab: تدريب AraBERT الحقيقي من البيانات الحالية

هذه الخلايا **لا تستخدم Google Drive**. الملف المضغوط يحتوي أيضاً على نماذج صغيرة PyTorch CPU **مدربة مسبقاً داخل هذه الحزمة على Silver** لتشغيل العرض محلياً؛ أما AraBERT فيحتاج تنزيل النموذج الأصلي من الإنترنت وتخصيصه فعلياً على GPU. **لا تُنسب إلى Gold البشري أي نتيجة**.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, sys, subprocess
uploaded=files.upload()
archives=[Path(x) for x in uploaded if x.lower().endswith('.zip')]
assert len(archives)==1, 'ارفع أرشيف المشروع النهائي ZIP'
zip_path=archives[0]
with ZipFile(zip_path) as z:
    for member in z.infolist():
        target=(Path('/content') / member.filename).resolve()
        if not target.is_relative_to(Path('/content')):
            raise ValueError('Unsafe file in ZIP: '+member.filename)
    z.extractall('/content')
root=Path('/content/LMI_PROJECT')
assert (root/'src/common.py').is_file(), 'ZIP must contain the top-level LMI_PROJECT folder'
os.chdir(root)
print('Loaded:',root)


In [ ]:
def run(*args):
    subprocess.run(list(map(str,args)),cwd=str(root),check=True)
run(sys.executable,'-m','pip','install','-r','requirements_runtime.txt','-r','requirements_dl.txt')
import torch
assert torch.cuda.is_available(), 'فعّل T4 GPU في Runtime > Change runtime type'
print('GPU:',torch.cuda.get_device_name(0))
run(sys.executable,'-m','scripts.verify_delivery')  # WARN is expected for missing human Gold
run(sys.executable,'-m','scripts.verify_model_artifacts')


## تدريب النماذج الثلاثة (Fine-tuning)

ستُحفظ الأوزان داخل `models_dl/classification/*` و`models_dl/ner/`. شغّل هذه الخلية لتحويل نماذج CPU التجريبية إلى نماذج AraBERT حقيقية تستخدمها الواجهة **تلقائياً فقط عند اكتمال ملفات الأوزان والـ Tokenizer**. لا يتم التظاهر بأن النموذج درّب إذا فشل الأمر، و`check=True` يوقف الدفتر عند الخطأ.


In [ ]:
for task in ('opportunity_type','geographic_scope'):
    modeldir=root/'models_dl'/'classification'/task
    present=(modeldir/'config.json').exists() and any((modeldir/x).exists() for x in
        ('model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json'))
    if not present:
        run(sys.executable,'-m','src.training.train_classifier','--task',task)
    else:
        print('Checkpoint exists, verifying by evaluation:',task)
    run(sys.executable,'-m','src.evaluation.evaluate_classifier','--task',task)
ner=root/'models_dl/ner'
present=(ner/'config.json').exists() and any((ner/x).exists() for x in
    ('model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json'))
if not present:
    run(sys.executable,'-m','src.training.train_ner')
run(sys.executable,'-m','src.evaluation.evaluate_ner')
print('Trained AraBERT checkpoints and SILVER-test evaluations saved.')


## الاختبارات بعد التدريب

تُنفذ الاختبارات كاملة. لا تُعدّ أي مرحلة ناجحة إذا فشل اختبار. قد تكشف الاختبارات الإضافية أخطاء تنفيذ بيئية تحتاج إلى إصلاح، ولن يقوم الدفتر بإخفائها.


In [ ]:
run(sys.executable,'-m','pytest','-q')
run(sys.executable,'-m','scripts.verify_delivery')
run(sys.executable,'-m','scripts.verify_model_artifacts')


## تشغيل تطبيق Streamlit (اختياري)

التطبيق يعمل على منفذ 8501 ضمن جلسة Colab؛ من أجل عرض محلي مستقر أمام اللجنة يُفضَّل تشغيله على الحاسوب بعد تنزيل النسخة. لا يوجد هنا رابط خارجي وهمي.


In [ ]:
import subprocess
app_process=subprocess.Popen([sys.executable,'-m','streamlit','run','app_dl/main.py',
    '--server.port=8501','--server.headless=true'],cwd=str(root))
print('Streamlit process PID:',app_process.pid,' port:8501')


## حفظ المشروع مع أوزان AraBERT قبل إغلاق Colab

بعد التدريب تُحمَّل نسخة كاملة، ويمكن تشغيلها محلياً بـ `python -m streamlit run app_dl/main.py`. لا تدّعِ توافر Gold قبل مراجعة بشرية موثقة.


In [ ]:
from shutil import make_archive
from google.colab import files
result=make_archive('/content/LMI_WITH_ARABERT','zip',root_dir='/content',base_dir='LMI_PROJECT')
print('Archive size:',Path(result).stat().st_size)
files.download(result)
